# District LLM Fine-Tuning

This notebook fine-tunes the district coordinator on the chat-style dataset generated by `scripts/generate_medium_district_dataset.py` and evaluates outputs with the richer `target_intersections` metrics from `district_llm.eval`.


## 1. Install dependencies

Run this first in Colab after cloning or uploading the repository.


In [ ]:
%%capture
import os
import re
import sys
from pathlib import Path

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth transformers datasets trl peft accelerate sentencepiece protobuf
else:
    import torch
    version = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(version, '0.0.34')
    !pip install sentencepiece protobuf "datasets>=2.20.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth transformers

REPO_ROOT = Path('/content/traffic-llm')
if REPO_ROOT.exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 2. Configure paths

Point these to the uploaded dataset and desired output adapter directory.


In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/content/outputs/district_llm_dataset_v2')
TRAIN_JSONL = DATASET_ROOT / 'train.jsonl'
VAL_JSONL = DATASET_ROOT / 'val.jsonl'
METADATA_JSON = DATASET_ROOT / 'metadata.json'
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
OUTPUT_DIR = Path('/content/outputs/district_llm_adapter_v2')
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
SEED = 3407
DEBUG_EXAMPLES = 10
MAX_EVAL_EXAMPLES = 200
RESTRICT_TARGETS_TO_VISIBLE_SUMMARY = False


## 3. Inspect dataset metadata and preview rows

This checks that the generated dataset has the expected diversity and shape before training.


In [ ]:
import json
from datasets import load_dataset

metadata = json.loads(METADATA_JSON.read_text())
print(json.dumps(metadata, indent=2)[:4000])

dataset = load_dataset(
    'json',
    data_files={'train': str(TRAIN_JSONL), 'val': str(VAL_JSONL)},
)
print(dataset)
print(json.dumps(dataset['train'][0], indent=2)[:2000])


## 4. Load the base model with Unsloth


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)


## 5. Convert chat rows into training text

We preserve the existing `messages` format and render it through the tokenizer chat template.


In [ ]:
EOS_TOKEN = tokenizer.eos_token or ''

def format_chat_examples(batch):
    texts = []
    for messages in batch['messages']:
        if hasattr(tokenizer, 'apply_chat_template'):
            rendered = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        else:
            rendered = '
'.join(f"{msg['role']}: {msg['content']}" for msg in messages)
        texts.append(rendered + EOS_TOKEN)
    return {'text': texts}

formatted_dataset = dataset.map(
    format_chat_examples,
    batched=True,
    remove_columns=dataset['train'].column_names,
)
formatted_dataset


## 6. Train with Unsloth SFT

Start with a smoke-test `max_steps`, then raise it for the real run.


In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset['train'],
    eval_dataset=formatted_dataset['val'],
    dataset_text_field='text',
    args=SFTConfig(
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=20,
        max_steps=300,
        learning_rate=2e-4,
        logging_steps=5,
        eval_steps=50,
        save_steps=50,
        optim='adamw_8bit',
        weight_decay=0.001,
        lr_scheduler_type='cosine',
        seed=SEED,
        output_dir=str(OUTPUT_DIR),
        report_to='none',
    ),
)


In [ ]:
trainer_stats = trainer.train()

## 7. Save the adapter


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))


## 8. Quick inference sanity check


In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
sample_messages = dataset['val'][0]['messages'][:2]
if hasattr(tokenizer, 'apply_chat_template'):
    prompt = tokenizer.apply_chat_template(sample_messages, tokenize=False, add_generation_prompt=True)
else:
    prompt = '
'.join(f"{msg['role']}: {msg['content']}" for msg in sample_messages) + '
assistant:'

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, max_new_tokens=128, do_sample=False, streamer=streamer)


## 9. Offline evaluation with improved `target_intersections` metrics

This reuses the helpers in `district_llm.eval` so the notebook and CLI stay aligned.


In [ ]:
from district_llm.eval import (
    DistrictTopologyIndex,
    evaluate_rows,
    load_rows,
    print_debug_examples,
)

val_rows = load_rows(VAL_JSONL, max_examples=MAX_EVAL_EXAMPLES)
topology_index = DistrictTopologyIndex('/content/traffic-llm/data/generated')
results = evaluate_rows(
    rows=val_rows,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    topology_index=topology_index,
    restrict_targets_to_visible_summary=RESTRICT_TARGETS_TO_VISIBLE_SUMMARY,
    debug_examples=DEBUG_EXAMPLES,
)

summary = {key: value for key, value in results.items() if key != 'debug_examples'}
print(json.dumps(summary, indent=2, sort_keys=True))


## 10. Inspect debug examples and target failure modes

These examples help determine whether poor `target_intersections` scores come from order sensitivity, partial overlap, invalid ids, or genuinely bad predictions.


In [ ]:
print_debug_examples(results['debug_examples'])

failure_buckets = results.get('target_intersections_failure_buckets', {})
print('Failure buckets:')
print(json.dumps(failure_buckets, indent=2, sort_keys=True))

if 'target_intersections_restricted_to_visible_summary' in results:
    print('Restricted-to-visible-summary metrics:')
    print(json.dumps(results['target_intersections_restricted_to_visible_summary'], indent=2, sort_keys=True))


## 11. Optional: evaluate the saved adapter from disk

Use this if you restart the runtime and want to score the saved adapter directly.


In [ ]:
# Example CLI equivalent inside the notebook runtime:
# !python -m district_llm.eval #   --model-path /content/outputs/district_llm_adapter_v2 #   --val-jsonl /content/outputs/district_llm_dataset_v2/val.jsonl #   --max-examples 200 #   --debug-examples 10
